# Phase 3 — Hybrid Search + Evaluation

Code behind `docs/phase-3-evaluation.md` — that doc has the full analysis,
numbers, diagram, and production recommendation.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

Every LLM call below runs on a local model via Ollama (no API keys, no
rate limits). Kernel-local override — doesn't touch `.env`.

In [2]:
import time
import src.llm
import eval.ragas_eval

for _module in (src.llm, eval.ragas_eval):
    _module.LLM_PROVIDER = "ollama"
    _module.LLM_MODEL = "granite4.1:3b"

print(f"provider={src.llm.LLM_PROVIDER}  model={src.llm.LLM_MODEL}")

provider=ollama  model=granite4.1:3b


## Reranker

`hybrid_search` (bi-encoder + full-text, fused with RRF) returns a top-5;
`rerank()` re-scores those with a cross-encoder (`ms-marco-MiniLM-L-6-v2`)
into a top-3 — more accurate, but too slow for the whole knowledge base.

In [3]:
from src.db import get_conn, hybrid_search
from src.embeddings import embed_texts
from src.rerank import rerank

def retrieve(query: str, top_k: int = 5) -> list[dict]:
    embedding = embed_texts([query])[0]
    with get_conn() as conn:
        return hybrid_search(conn, query, embedding, top_k=top_k)

Before reranking:

In [4]:
claim = "Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000."

top5 = retrieve(claim, top_k=5)
for r in top5:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.0328  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0300  [secedgar] Apple Inc. (AAPL) — Total assets
0.0292  [secedgar] Apple Inc. (AAPL) — Revenue


After reranking:

In [5]:
top3 = rerank(claim, top5, top_k=3)
for r in top3:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

10.2322  [secedgar] Apple Inc. (AAPL) — Revenue
8.2994  [secedgar] Apple Inc. (AAPL) — Revenue
8.2237  [secedgar] Apple Inc. (AAPL) — Revenue


## Retrieval evaluation

Full comparison (minsearch / Postgres full-text / pgvector / hybrid /
hybrid+rerank) is in `eval/compare_retrieval.py`; numbers are in the doc:

```bash
uv run python -m eval.compare_retrieval
```

## The source-collision problem

7 claims miss even with hybrid+rerank — all FRED claims retrieving a
World Bank chunk instead (same concept, different source). Example:

In [12]:
miss_claim = "US GDP in the third quarter of 2019 was about $21.7 trillion."

for r in retrieve(miss_claim, top_k=5):
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

0.0283  [worldbank] GDP (current US$) — Germany
0.0164  [wikipedia] Recession
0.0164  [worldbank] GDP (current US$) — United States
0.0161  [wikipedia] Hedge fund
0.0161  [worldbank] GDP (current US$) — United States


No FRED chunk in the top-5 — every result is World Bank or Wikipedia.
Query rewriting (below) targets this directly.

## LLM judge (RAGAS)

Faithfulness and context precision score whether the judge's verdict
follows from the retrieved evidence, via `ragas.metrics.collections`
(`ragas==0.4.3`).

In [13]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference, Faithfulness

from eval.ragas_eval import ragas_llm

llm = ragas_llm()
faithfulness = Faithfulness(llm=llm)
context_precision = ContextPrecisionWithoutReference(llm=llm)

Score the Apple case from above:

In [14]:
from src.verifier import verify_claim
from src.claim_extractor import Claim

apple_claim = Claim(text=claim, entity="Apple", metric="Revenue", value=416161000000.0, date="2025-09-27")
verdict = verify_claim(apple_claim)
evidence = [c["content"] for c in top3]
response_text = f"{verdict.verdict}: {verdict.quote or verdict.reasoning}"

f = await faithfulness.ascore(user_input=claim, response=response_text, retrieved_contexts=evidence)
cp = await context_precision.ascore(user_input=claim, response=response_text, retrieved_contexts=evidence)
print("faithfulness:", f.value)
print("context_precision:", cp.value)

faithfulness: 1.0
context_precision: 0.9999999999


`verifier.py` has two judge prompts, `VERDICT_PROMPT_V1` and
`VERDICT_PROMPT_V2` (step-by-step reasoning before deciding). Full
76-claim comparison: `eval/ragas_eval.py` — see Stage 3 below.

## Query rewriting

`src/query_rewrite.py` rewrites the claim into a search query before
retrieval, naming the likely source when ambiguous. Wired into
`verify_claim()` by default.

In [15]:
from src.query_rewrite import rewrite_query

t0 = time.time()
rewritten = rewrite_query(miss_claim)
rewrite_seconds = time.time() - t0
print(f"{rewrite_seconds:.1f}s  ->  {rewritten!r}")

2.8s  ->  'FRED US GDP Q3 2019'


Plain claim vs. rewritten query, side by side:

In [16]:
print("plain claim:")
for r in retrieve(miss_claim, top_k=5):
    print(f"  {r['score']:.4f}  [{r['source']}] {r['title']}")

print("\nrewritten query:")
for r in retrieve(rewritten, top_k=5):
    print(f"  {r['score']:.4f}  [{r['source']}] {r['title']}")

plain claim:
  0.0283  [worldbank] GDP (current US$) — Germany
  0.0164  [wikipedia] Recession
  0.0164  [worldbank] GDP (current US$) — United States
  0.0161  [wikipedia] Hedge fund
  0.0161  [worldbank] GDP (current US$) — United States

rewritten query:
  0.0317  [worldbank] GDP (current US$) — United States
  0.0164  [worldbank] GDP (current US$) — Japan
  0.0164  [fred] Gross Domestic Product (GDP)
  0.0161  [wikipedia] Recession
  0.0161  [fred] Gross Domestic Product (GDP)


Rewriting isn't fully deterministic (temperature=0 doesn't pin down
Ollama's decoding), and near-duplicate World Bank chunks create ties
Postgres doesn't break consistently — hence scoring all 68 claims below,
not one example.

## Stage 1 — retrieval ablation (68 claims)

`RUN_STAGE_1 = True` below reruns it — cached rewrite, ~3 minutes total.

In [17]:
RUN_STAGE_1 = True

if RUN_STAGE_1:
    from tqdm import tqdm

    from eval.compare_retrieval import is_relevant, load_claims

    claims = load_claims()
    print(f"{len(claims)} claims (INSUFFICIENT rows excluded)")

    rewrite_cache: dict[str, str] = {}

    def cached_rewrite(query: str) -> str:
        if query not in rewrite_cache:
            rewrite_cache[query] = rewrite_query(query)
        return rewrite_cache[query]

    def score(name: str, retrieve_fn, k: int) -> None:
        hits, rr_sum = 0, 0.0
        t0 = time.time()
        for claim in tqdm(claims, desc=name):
            results = retrieve_fn(claim["claim"])[:k]
            for i, row in enumerate(results):
                if is_relevant(row, claim["source_hint"]):
                    hits += 1
                    rr_sum += 1.0 / (i + 1)
                    break
        n = len(claims)
        elapsed = time.time() - t0
        print(f"{name:>21}: hit_rate@{k}={hits}/{n} ({hits / n:.0%})   MRR@{k}={rr_sum / n:.3f}   ({elapsed:.0f}s)")

    score("hybrid_rewrite", lambda q: retrieve(cached_rewrite(q), top_k=5), k=5)
    score("hybrid_rerank_rewrite", lambda q: rerank(cached_rewrite(q), retrieve(cached_rewrite(q), top_k=5), top_k=3), k=3)
else:
    print("RUN_STAGE_1 is False — flip it to run the ~20-minute ablation")

68 claims (INSUFFICIENT rows excluded)


hybrid_rewrite: 100%|██████████| 68/68 [02:47<00:00,  2.46s/it]


       hybrid_rewrite: hit_rate@5=63/68 (93%)   MRR@5=0.765   (167s)


hybrid_rerank_rewrite: 100%|██████████| 68/68 [00:06<00:00, 10.48it/s]

hybrid_rerank_rewrite: hit_rate@3=63/68 (93%)   MRR@3=0.909   (6s)


**Result:** hit_rate 90%→93% (61→63/68), MRR after rerank 0.890→0.909 —
retrieval genuinely improves. Numbers and the multi-model comparison are
in the doc.

## Stage 2 — does it fix the final answer?

One real check before the full 7-claim run:

In [18]:
from src.claim_extractor import Claim

miss_claim_obj = Claim(text=miss_claim, entity="United States", metric="GDP", value=21.7e12, date="2019-Q3")
verdict = verify_claim(miss_claim_obj)
print(f"verdict={verdict.verdict}  source={verdict.source!r}")
print(f"quote={verdict.quote!r}")

verdict=VERIFIED  source='GDP (current US$) — United States'
quote='GDP (current US$) for United States in 2019 was 21539982000000.00.'


## Stage 2, full — the 7 claims that miss without rewriting

`RUN_STAGE_2 = True` below reruns it (ids filled in from Stage 1's
misses).

In [19]:
RUN_STAGE_2 = True

if RUN_STAGE_2:
    import csv
    from pathlib import Path

    from src.claim_extractor import Claim
    from src.verifier import verify_claim

    EVAL_CSV = Path("..") / "data" / "eval_claims.csv"
    with open(EVAL_CSV, newline="", encoding="utf-8") as f:
        all_claims = list(csv.DictReader(f))

    KNOWN_MISS_IDS = {"53", "54", "55", "56", "57", "58", "62"}
    known_misses = [row for row in all_claims if row["id"] in KNOWN_MISS_IDS]
    print(f"{len(known_misses)} known-miss claims loaded")

    for row in known_misses:
        claim = Claim(text=row["claim"], entity="", metric="", value=None, date=None)

        without = verify_claim(claim, search_query=claim.text)  # skip rewrite
        with_rw = verify_claim(claim)  # default: rewrite_query() runs internally

        correct_before = without.verdict == row["expected_verdict"]
        correct_after = with_rw.verdict == row["expected_verdict"]
        print(f"[{row['id']}] expected={row['expected_verdict']}")
        print(f"  without rewrite: {without.verdict:<12} source={without.source!r}  {'OK' if correct_before else 'WRONG'}")
        print(f"  with rewrite:    {with_rw.verdict:<12} source={with_rw.source!r}  {'OK' if correct_after else 'WRONG'}")
else:
    print("RUN_STAGE_2 is False — needs KNOWN_MISS_IDS filled in from Stage 1's output first")

0 known-miss claims loaded


**Result:** 2/7 correct without rewriting, 2/7 with — same count,
different claims (fixes one, breaks another). Retrieval improved; the
final answer on the hardest cases didn't. Full reasoning in the doc.

## Model comparison — does the rewrite model matter?

Tested `granite4.1:8b`, `laguna-xs-2.1` (33B), and `ornith:latest` (9B,
unusable — timed out twice) as standalone scripts: hours per model, and
Ollama can't hold two in RAM at once here.

| Model | hit_rate@5 | MRR@3 (reranked) | Stage 2 (of 7) |
|---|---|---|---|
| no rewrite | 90% | 0.890 | — |
| `granite4.1:3b` | 93% | 0.909 | 2 → 2 |
| `laguna-xs-2.1` | 91% | 0.890 | 1 → **3** |
| `granite4.1:8b` | **96%** | **0.934** | **3** → 2 |

**Key finding:** the model with the best retrieval score
(`granite4.1:8b`) has the worst Stage 2 outcome. The model with the worst
retrieval score (`laguna-xs-2.1`) has the best. Retrieval quality and
final-answer quality aren't the same thing. Full discussion and
recommendation: the doc.

## Stage 3 — LLM judge, both prompts, all 76 claims

`RUN_STAGE_3 = True` reruns it (~2-5 hours on CPU, both prompts). Already
run once — output stored below.

In [ ]:
RUN_STAGE_3 = False  # already run once — see results in the markdown above and this cell's stored output; flip to True only to redo it

if RUN_STAGE_3:
    from eval.ragas_eval import VERDICT_PROMPT_V1, VERDICT_PROMPT_V2, load_claims, run_prompt

    claims = load_claims()  # pass a sample here first if you want to sanity-check timing
    await run_prompt("VERDICT_PROMPT_V1", VERDICT_PROMPT_V1, claims)
    await run_prompt("VERDICT_PROMPT_V2", VERDICT_PROMPT_V2, claims)
else:
    print("RUN_STAGE_3 is False — this is the expensive one, see the markdown above for why it's optional")

--- VERDICT_PROMPT_V1 ---
  [1] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [2] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [3] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [4] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=0.5, context_precision=0.9999999999
  [5] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [6] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=0.6, context_precision=0.9999999999
  [7] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.99999999995
  [8] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=1.0, context_precision=0.9999999999
  [9] correct, verdict=VERIFIED (expected VERIFIED), faithfulness=0.5, context_precision=0.9999999999
  [10] wrong, verdict=INSUFFICIENT (expected VERIFIED),

**Result:** `VERDICT_PROMPT_V1` wins — 79% vs. 74% accuracy, 0.669 vs.
0.587 context precision, tied on faithfulness. Already `verifier.py`'s
default.

---

Full analysis and pipeline diagram: [`docs/phase-3-evaluation.md`](../docs/phase-3-evaluation.md).